# ClickTheLook — HPC Inference Pipeline

This notebook runs the full clothing inference pipeline on a video file. Starting from a pre-trained YOLO model, it detects clothing items frame by frame, tracks each item across the video using DeepSORT, applies global re-identification to keep a stable ID on each item even if it briefly leaves the frame, selects the sharpest crop for each unique clothing detection, filters out blurry and duplicate crops, and writes structured output metadata with timestamps.

Run all cells from top to bottom. The only cell you need to edit before running is **Step 2 (Paths and Settings)**.

Note: The notebook requires a GPU for quick processing.

## Step 1 — Install Required Packages

Before anything can run, all the software the pipeline depends on needs to be installed. This cell reads the `requirements.txt` file that lives alongside this notebook and installs every package listed in it automatically. If the file is not found, it falls back to installing the essential packages individually.

Run this once per environment. It is safe to re-run if you are unsure.

In [ ]:
import subprocess, sys
from pathlib import Path

req_file = Path.cwd() / "requirements.txt"

if req_file.exists():
    print(f"Installing packages from {req_file.name} ...")
    subprocess.check_call([sys.executable, "-m", "pip", "install", "-r", str(req_file), "-q"])
    print(f"All packages installed from requirements.txt.")
else:
    print("requirements.txt not found in current directory. Installing essential packages individually ...")
    pkgs = ["ultralytics", "deep-sort-realtime", "opencv-python-headless",
            "numpy", "matplotlib", "mlflow"]
    for p in pkgs:
        subprocess.check_call([sys.executable, "-m", "pip", "install", p, "-q"])
    print("Essential packages installed.")

print("\nStep 1 complete. Proceed to Step 2 to set your paths.")

## Step 2 — Set Your Paths and Settings

This is the only cell you need to edit. Fill in the path to your input video, the path to your trained YOLO model file, and where you want the outputs saved. The thresholds below control how the detector and tracker behave. The defaults are pre-tuned and work well for most videos, so leave them as-is unless you want to experiment.

In [ ]:
from pathlib import Path

VIDEO_PATH      = "./your/video/path"       # input video
MODEL_PATH      = "./your/model/path"         # trained YOLO weights
SAVE_VIDEO      = "your/output/video/path"       # annotated output video (None to skip)
OUTPUT_BASE_DIR = "./your/output/directory"         # logs/, detections/, output/ created here

CONF = 0.5
IOU  = 0.45

DEEPSORT_MAX_AGE    = 5
DEEPSORT_N_INIT     = 3
DEEPSORT_MAX_COSINE = 0.3
DEEPSORT_EMBEDDER   = "mobilenet"

USE_GLOBAL_IDS       = True
GID_MAX_GAP_S        = 43.0
GID_COSINE_THRESHOLD = 0.15
GID_SPATIAL_GATE_PPS = 50.0

BLUR_THRESHOLD  = 90.0   # Laplacian variance; below = blurry
PHASH_THRESHOLD = 13     # dHash Hamming distance; below = near-duplicate (0-64)

MIN_VISIBLE_S  = 2.8     # exclude identities visible less than this
LINE_THICKNESS = 2

MLFLOW_TRACKING_URI = "./your/mlflow/server"       # or "http://your-mlflow-server:5000"
MLFLOW_EXPERIMENT   = "ClickTheLook"

LOG_DIR        = Path(OUTPUT_BASE_DIR) / "logs"
DETECTIONS_DIR = Path(OUTPUT_BASE_DIR) / "detections"
OUTPUT_DIR     = Path(OUTPUT_BASE_DIR) / "output"
for _d in [LOG_DIR, DETECTIONS_DIR, OUTPUT_DIR]:
    _d.mkdir(parents=True, exist_ok=True)

print("Configuration set.")
print(f"  Video input : {VIDEO_PATH}")
print(f"  Model       : {MODEL_PATH}")
print(f"  Video out   : {SAVE_VIDEO}")
print(f"  Logs        : {LOG_DIR}")
print(f"  Detections  : {DETECTIONS_DIR}")
print(f"  Output JSON : {OUTPUT_DIR}")
print(f"  MLflow      : {MLFLOW_TRACKING_URI}  (experiment: {MLFLOW_EXPERIMENT})")
print("\nStep 2 complete. Proceed to Step 3 to check your hardware.")

## Step 3 — Check Available Hardware

The pipeline checks whether a GPU is available on the machine. A GPU can process video many times faster than a CPU. If one is found it will be used automatically for both the detection model and the appearance embedder inside the tracker. If not, processing falls back to CPU.

In [ ]:
import torch

if torch.cuda.is_available():
    DEVICE       = "0"
    EMBEDDER_GPU = True
    props        = torch.cuda.get_device_properties(0)
    print(f"GPU found  : {props.name}")
    print(f"VRAM       : {props.total_memory / 1e9:.1f} GB")
    print(f"CUDA       : {torch.version.cuda}")
else:
    DEVICE       = "cpu"
    EMBEDDER_GPU = False
    print("No CUDA GPU detected — running on CPU (processing will be slower)")

print(f"\nDevice set : {DEVICE}")
print("Step 3 complete. Proceed to Step 4.")

## Step 4 — Load Clothing Category Labels

The detection model can recognise 13 types of clothing. This cell maps each clothing type to the number the model uses internally. When the model says it detected class 2, this lookup translates that to "long sleeve top" so all the output labels are human-readable.

In [ ]:
CATEGORIES = {
    1:  "short_sleeve_top",     2:  "long_sleeve_top",
    3:  "short_sleeve_outwear", 4:  "long_sleeve_outwear",
    5:  "vest",                 6:  "sling",
    7:  "shorts",               8:  "trousers",
    9:  "skirt",                10: "short_sleeve_dress",
    11: "long_sleeve_dress",    12: "vest_dress",
    13: "sling_dress",
}
print(f"{len(CATEGORIES)} clothing categories loaded.")
print("Step 4 complete. Proceed to Step 5.")

## Step 5 — Load Libraries and Utility Functions

This cell imports all the Python libraries the pipeline relies on and defines several small helper functions used throughout. These helpers handle tasks such as measuring how sharp or blurry a crop image is, generating a visual fingerprint of an image to detect near-duplicate crops, and converting frame numbers into human-readable timestamps like "00:01:23.500".

In [ ]:
import cv2
import json
import mlflow
import numpy as np
import platform
import socket
import time
from datetime import datetime
from random import randint


def _cosine_dist(a: np.ndarray, b: np.ndarray) -> float:
    na, nb = np.linalg.norm(a), np.linalg.norm(b)
    if na == 0 or nb == 0:
        return 1.0
    return 1.0 - float(np.dot(a, b) / (na * nb))


def _blur_score(img_bgr: np.ndarray) -> float:
    """Laplacian variance — higher = sharper."""
    gray = cv2.cvtColor(img_bgr, cv2.COLOR_BGR2GRAY)
    return float(cv2.Laplacian(gray, cv2.CV_64F).var())


def _dhash(img_bgr: np.ndarray, size: int = 8) -> int:
    """64-bit perceptual fingerprint: encodes relative brightness changes between adjacent pixels."""
    gray    = cv2.cvtColor(img_bgr, cv2.COLOR_BGR2GRAY)
    resized = cv2.resize(gray, (size + 1, size), interpolation=cv2.INTER_AREA)
    diff    = resized[:, 1:] > resized[:, :-1]
    return sum(1 << i for i, v in enumerate(diff.flatten()) if v)


def _frame_to_ts(frame_idx: int, fps: float) -> str:
    total_s = frame_idx / max(fps, 1e-6)
    hours   = int(total_s // 3600)
    minutes = int((total_s % 3600) // 60)
    seconds = total_s % 60
    return f"{hours:02d}:{minutes:02d}:{seconds:06.3f}"


def _duration_text(seconds: float) -> str:
    m = int(seconds // 60)
    s = int(seconds % 60)
    return f"{m}m {s}s" if m > 0 else f"{s}s"


def _arr_stats(arr) -> dict:
    if not arr:
        return {"avg": 0.0, "min": 0.0, "max": 0.0, "p50": 0.0, "p95": 0.0}
    a = np.array(arr, dtype=float)
    return {
        "avg": round(float(np.mean(a)),           3),
        "min": round(float(np.min(a)),            3),
        "max": round(float(np.max(a)),            3),
        "p50": round(float(np.percentile(a, 50)), 3),
        "p95": round(float(np.percentile(a, 95)), 3),
    }


print("Libraries imported and helper functions defined.")
print("Step 5 complete. Proceed to Step 6.")

## Step 6 — Set Up the Global Re-Identification System

The tracker works frame by frame, but a clothing item can temporarily leave the frame or become occluded and then reappear. When that happens, the tracker treats it as a brand new detection and assigns it a fresh ID, breaking the continuity of the track. This component prevents that. After each frame it compares every newly detected clothing item against recently seen ones, looking at both where the item is in the frame and what it looks like visually. If the match is strong enough, the original ID is restored instead of starting a new one, so each unique clothing item carries one consistent ID throughout the entire video.

In [ ]:
class GlobalIdentityManager:
    """
    Re-identifies tracked objects across disappearance gaps.
    Maintains a stable global ID per physical object even when the tracker
    drops and re-acquires the track.
    """

    def __init__(self, max_gap_s: float, cosine_threshold: float, spatial_gate_pps: float):
        self.max_gap_s        = max_gap_s
        self.cosine_threshold = cosine_threshold
        self.spatial_gate_pps = spatial_gate_pps
        self._next_gid        = 1
        self._active          = {}
        self._identities      = {}

    def update(self, tracked: np.ndarray, embeddings: dict, class_names: dict, timestamp: float) -> dict:
        """Call once per frame after update_tracker(). Returns {tracker_id: global_id}."""
        self._expire(timestamp)
        current_tids = set()
        id_map       = {}

        for row in tracked:
            tid  = int(row[8])
            cid  = int(row[4])
            name = class_names.get(cid + 1, str(cid))
            cx   = (float(row[0]) + float(row[2])) / 2.0
            cy   = (float(row[1]) + float(row[3])) / 2.0
            emb  = embeddings.get(tid)
            current_tids.add(tid)

            if tid in self._active:
                gid = self._active[tid]
                self._refresh(gid, emb, name, (cx, cy), timestamp)
            else:
                gid = self._match(name, (cx, cy), emb, timestamp)
                if gid is None:
                    gid = self._next_gid
                    self._next_gid += 1
                self._active[tid] = gid
                self._refresh(gid, emb, name, (cx, cy), timestamp)
            id_map[tid] = gid

        for tid in list(self._active):
            if tid not in current_tids:
                del self._active[tid]
        return id_map

    def _refresh(self, gid, emb, class_name, center, ts):
        entry = self._identities.get(gid)
        if entry is None:
            self._identities[gid] = {"emb": emb, "class_name": class_name, "center": center, "last_ts": ts}
        else:
            if emb is not None:
                prev = entry["emb"]
                entry["emb"] = emb if prev is None else (0.7 * prev + 0.3 * emb)
            entry["center"]  = center
            entry["last_ts"] = ts

    def _expire(self, now):
        active_gids = set(self._active.values())
        for gid in [g for g, e in self._identities.items()
                    if g not in active_gids and (now - e["last_ts"]) > self.max_gap_s]:
            del self._identities[gid]

    def _match(self, class_name, center, emb, timestamp):
        active_gids = set(self._active.values())
        best_gid    = None
        best_score  = self.cosine_threshold
        for gid, entry in self._identities.items():
            if gid in active_gids:
                continue
            if entry["class_name"] != class_name:
                continue
            gap = timestamp - entry["last_ts"]
            if gap > self.max_gap_s:
                continue
            ex, ey   = entry["center"]
            dist_px  = ((center[0] - ex) ** 2 + (center[1] - ey) ** 2) ** 0.5
            if dist_px > self.spatial_gate_pps * max(gap, 0.1):
                continue
            if emb is None or entry["emb"] is None:
                if gap < 2.0 and best_gid is None:
                    best_gid = gid
                continue
            score = _cosine_dist(emb, entry["emb"])
            if score < best_score:
                best_score = score
                best_gid   = gid
        return best_gid


print("Global Re-ID system ready. It will maintain stable IDs for each person across the whole video.")
print("Step 6 complete. Proceed to Step 7.")

## Step 7 — Set Up the Run Logger

Every run of the pipeline produces a detailed log. This component silently collects statistics on every frame: how many clothing items were detected, how long each processing step took, which clothing categories appeared, and how long each unique clothing item was visible in the video. When the pipeline finishes, it compiles everything into a JSON log file saved to the logs folder so you have a full record of the run.

In [ ]:
_TIMELINE_SAMPLE_INTERVAL = 30

try:
    import psutil as _psutil
    _RAM_GB   = round(_psutil.virtual_memory().total / 1e9, 1)
except Exception:
    _RAM_GB = None

_TORCH_VERSION       = torch.__version__
_CUDA_AVAILABLE      = torch.cuda.is_available()
_GPU_NAME            = torch.cuda.get_device_name(0) if _CUDA_AVAILABLE else None

try:
    import ultralytics as _ul
    _ULTRALYTICS_VERSION = _ul.__version__
except Exception:
    _ULTRALYTICS_VERSION = "unknown"


def extract_model_info(model) -> dict:
    info = {
        "task":        getattr(model, "task", "detect"),
        "num_classes": len(getattr(model, "names", {})),
        "class_names": getattr(model, "names", {}),
    }
    try:
        result = model.info(verbose=False)
        if isinstance(result, (list, tuple)) and len(result) >= 2:
            info["num_layers"] = int(result[0])
            info["num_params"] = int(result[1])
    except Exception:
        pass
    return info


class RunLogger:
    """Collects per-frame stats and writes a JSON summary on finalize()."""

    def __init__(self, *, model_path, model_info, tracker_backend, tracker_config,
                 source, source_meta, conf, iou, device, save_video, log_dir):
        self.run_id    = datetime.now().strftime("%Y%m%d_%H%M%S")
        self._t_start  = time.time()
        self._src_fps  = max(float(source_meta.get("source_fps", 30.0)), 1e-6)
        self._log_dir  = Path(log_dir)

        self._inference_ms      = []
        self._tracker_ms        = []
        self._draw_ms           = []
        self._dets_per_frame    = []
        self._tracks_per_frame  = []
        self._conf_scores       = []
        self._class_det_counts  = {}
        self._class_trk_counts  = {}
        self._track_registry    = {}
        self._gid_current       = {}
        self._gid_intervals     = {}
        self._timeline          = []
        self._frame_anomalies   = []

        self._meta = {
            "run_id":           self.run_id,
            "timestamp_start":  datetime.now().isoformat(),
            "timestamp_end":    None,
            "total_duration_s": None,
            "system": {
                "hostname":            socket.gethostname(),
                "os":                  platform.system(),
                "torch_version":       _TORCH_VERSION,
                "ultralytics_version": _ULTRALYTICS_VERSION,
                "gpu_available":       _CUDA_AVAILABLE,
                "gpu_name":            _GPU_NAME,
                "ram_gb":              _RAM_GB,
            },
            "model":   {"path": str(model_path), "conf_threshold": conf,
                        "iou_threshold": iou, "device": str(device), **model_info},
            "tracker": {"backend": tracker_backend, **tracker_config},
            "source":  {"input": str(source), "output_video": str(save_video) if save_video else None,
                        **source_meta},
            "performance": {}, "detections": {}, "tracking": {},
            "anomalies": {}, "timeline": {},
        }

    def log_frame(self, frame_idx, inference_ms, tracker_ms, draw_ms,
                  detections, tracked, class_names, id_map=None):
        n_dets   = len(detections)
        n_tracks = len(tracked)
        total_ms = inference_ms + tracker_ms + draw_ms

        self._inference_ms.append(inference_ms)
        self._tracker_ms.append(tracker_ms)
        self._draw_ms.append(draw_ms)
        self._dets_per_frame.append(n_dets)
        self._tracks_per_frame.append(n_tracks)

        if n_dets > 0:
            self._conf_scores.extend(detections[:, 4].tolist())
            for det in detections:
                name = class_names.get(int(det[5]) + 1, f"cls_{int(det[5])}")
                self._class_det_counts[name] = self._class_det_counts.get(name, 0) + 1

        if n_tracks > 0:
            for row in tracked:
                tid  = int(row[8])
                name = class_names.get(int(row[4]) + 1, f"cls_{int(row[4])}")
                self._class_trk_counts[name] = self._class_trk_counts.get(name, 0) + 1
                if tid not in self._track_registry:
                    self._track_registry[tid] = {"class": name, "first_frame": frame_idx,
                                                  "last_frame": frame_idx, "gap_count": 0}
                else:
                    entry = self._track_registry[tid]
                    if frame_idx > entry["last_frame"] + 1:
                        entry["gap_count"] += frame_idx - entry["last_frame"] - 1
                    entry["last_frame"] = frame_idx

        gids_this_frame = set()
        if n_tracks > 0:
            for row in tracked:
                tid  = int(row[8])
                gid  = id_map.get(tid, tid) if id_map else tid
                name = class_names.get(int(row[4]) + 1, f"cls_{int(row[4])}")
                gids_this_frame.add(gid)
                if gid not in self._gid_current:
                    self._gid_current[gid] = {"start_frame": frame_idx,
                                               "last_frame": frame_idx, "class_name": name}
                else:
                    self._gid_current[gid]["last_frame"] = frame_idx

        for gid in list(self._gid_current):
            if gid not in gids_this_frame:
                self._close_interval(gid)

        if n_dets == 0 and frame_idx > 1:
            self._frame_anomalies.append({"frame": frame_idx, "type": "zero_detections",
                                           "inference_ms": round(inference_ms, 1)})
        if total_ms > 500:
            self._frame_anomalies.append({"frame": frame_idx, "type": "slow_frame",
                                           "total_ms": round(total_ms, 1)})

        if frame_idx % _TIMELINE_SAMPLE_INTERVAL == 0:
            inst_fps = round(1000.0 / total_ms, 1) if total_ms > 0 else 0.0
            self._timeline.append({"frame": frame_idx, "detections": n_dets,
                                    "tracks": n_tracks, "inference_ms": round(inference_ms, 1),
                                    "tracker_ms": round(tracker_ms, 1),
                                    "draw_ms": round(draw_ms, 1), "inst_fps": inst_fps})

    def _close_interval(self, gid):
        entry = self._gid_current.pop(gid, None)
        if entry is None:
            return
        start_f = entry["start_frame"]
        end_f   = entry["last_frame"]
        dur_s   = (end_f - start_f) / self._src_fps
        interval = {
            "start_frame":      start_f,
            "end_frame":        end_f,
            "start_ts":         _frame_to_ts(start_f, self._src_fps),
            "end_ts":           _frame_to_ts(end_f,   self._src_fps),
            "duration_seconds": round(dur_s, 3),
            "duration_text":    _duration_text(dur_s),
        }
        if gid not in self._gid_intervals:
            self._gid_intervals[gid] = {"class_name": entry["class_name"], "intervals": []}
        self._gid_intervals[gid]["intervals"].append(interval)

    def finalize(self, frames_processed, total_elapsed):
        m = self._meta
        m["timestamp_end"]    = datetime.now().isoformat()
        m["total_duration_s"] = round(total_elapsed, 2)

        total_ms_per_frame = [i + t + d for i, t, d in
                               zip(self._inference_ms, self._tracker_ms, self._draw_ms)]
        bottleneck = max(
            {"inference": np.mean(self._inference_ms) if self._inference_ms else 0,
             "tracker":   np.mean(self._tracker_ms)   if self._tracker_ms   else 0,
             "draw":      np.mean(self._draw_ms)       if self._draw_ms      else 0},
            key=lambda k: {"inference": np.mean(self._inference_ms) if self._inference_ms else 0,
                           "tracker":   np.mean(self._tracker_ms)   if self._tracker_ms   else 0,
                           "draw":      np.mean(self._draw_ms)       if self._draw_ms      else 0}[k]
        )

        m["performance"] = {
            "frames_processed":        frames_processed,
            "realtime_avg_fps":        round(frames_processed / total_elapsed, 2) if total_elapsed > 0 else 0,
            "inference_ms":            _arr_stats(self._inference_ms),
            "tracker_ms":              _arr_stats(self._tracker_ms),
            "draw_ms":                 _arr_stats(self._draw_ms),
            "total_pipeline_ms":       _arr_stats(total_ms_per_frame),
            "frames_with_zero_dets":   self._dets_per_frame.count(0),
            "frames_with_zero_tracks": self._tracks_per_frame.count(0),
            "zero_detection_rate":     round(self._dets_per_frame.count(0) / max(frames_processed, 1), 4),
            "bottleneck":              bottleneck,
        }
        m["detections"] = {
            "total":     sum(self._dets_per_frame),
            "per_frame": _arr_stats(self._dets_per_frame),
            "confidence": _arr_stats(self._conf_scores),
            "per_class_detections": dict(
                sorted(self._class_det_counts.items(), key=lambda x: x[1], reverse=True)),
            "per_class_tracked_frames": dict(
                sorted(self._class_trk_counts.items(), key=lambda x: x[1], reverse=True)),
        }
        lifespans     = [v["last_frame"] - v["first_frame"] + 1 for v in self._track_registry.values()]
        track_summary = sorted(
            [{"id": tid, "class": v["class"], "first_frame": v["first_frame"],
              "last_frame": v["last_frame"],
              "lifespan_frames": v["last_frame"] - v["first_frame"] + 1,
              "kalman_gap_frames": v["gap_count"]} for tid, v in self._track_registry.items()],
            key=lambda x: x["lifespan_frames"], reverse=True)
        m["tracking"] = {
            "total_unique_ids":       len(self._track_registry),
            "tracks_per_frame":       _arr_stats(self._tracks_per_frame),
            "track_lifespan_frames":  _arr_stats(lifespans) if lifespans else {},
            "longest_track":          track_summary[0]  if track_summary else None,
            "shortest_track":         track_summary[-1] if track_summary else None,
            "track_summary":          track_summary,
        }
        slow_frames = [a for a in self._frame_anomalies if a["type"] == "slow_frame"]
        m["anomalies"] = {"slow_frames_count": len(slow_frames), "slow_frames": slow_frames[:50]}
        m["timeline"]  = {"sample_interval_frames": _TIMELINE_SAMPLE_INTERVAL,
                           "samples": self._timeline}

        for gid in list(self._gid_current):
            self._close_interval(gid)
        identity_summary = []
        for gid, data in sorted(self._gid_intervals.items()):
            total_s = sum(iv["duration_seconds"] for iv in data["intervals"])
            identity_summary.append({
                "id":                    gid,
                "class":                 data["class_name"],
                "intervals":             data["intervals"],
                "total_visible_seconds": round(total_s, 3),
                "total_visible_text":    _duration_text(total_s),
            })
        m["identities"]       = identity_summary
        self.identity_summary = identity_summary

        self._log_dir.mkdir(parents=True, exist_ok=True)
        log_path = self._log_dir / f"{self.run_id}.json"
        with open(log_path, "w") as fh:
            json.dump(m, fh, indent=2)
        return str(log_path)


print("Run logger ready. All pipeline metrics will be recorded to the logs folder.")
print("Step 7 complete. Proceed to Step 8.")

## Step 8 — Set Up the Object Tracker

The DeepSORT tracker is the component that connects detections across frames into continuous tracks. For each detected bounding box, it combines two signals: a prediction of where the box should appear next (based on how it was moving) and a visual fingerprint of what the clothing item looks like (from a small neural network). Combining both signals keeps tracking robust even when items temporarily get occluded in the frame or overlap with each other. This cell also defines the function that draws labelled bounding boxes on each video frame.

In [ ]:
from deep_sort_realtime.deepsort_tracker import DeepSort

_TRACK_COLORS = [(randint(0, 255), randint(0, 255), randint(0, 255)) for _ in range(5000)]


class DeepSortWrapper:
    def __init__(self, tracker: DeepSort):
        self._tracker        = tracker
        self.centroid_history = {}

    def update(self, detections: np.ndarray, frame: np.ndarray) -> np.ndarray:
        raw = []
        for det in detections:
            x1, y1, x2, y2, conf, cls = det
            raw.append(([x1, y1, x2 - x1, y2 - y1], float(conf), int(cls)))
        tracks     = self._tracker.update_tracks(raw, frame=frame)
        results    = []
        active_ids = set()
        for track in tracks:
            if not track.is_confirmed():
                continue
            tid = int(track.track_id)
            active_ids.add(tid)
            x1, y1, x2, y2 = track.to_ltrb()
            cls = track.get_det_class() if track.get_det_class() is not None else 0
            cx, cy = (x1 + x2) / 2, (y1 + y2) / 2
            self.centroid_history.setdefault(tid, []).append((cx, cy))
            results.append([x1, y1, x2, y2, cls, 0.0, 0.0, 0.0, tid])
        for tid in list(self.centroid_history):
            if tid not in active_ids:
                del self.centroid_history[tid]
        return np.array(results, dtype=float) if results else np.empty((0, 9))

    def get_embeddings(self) -> dict:
        result = {}
        try:
            for track in self._tracker.tracker.tracks:
                if track.is_confirmed() and hasattr(track, "features") and track.features:
                    result[int(track.track_id)] = np.mean(track.features, axis=0)
        except Exception:
            pass
        return result


def init_tracker(max_age, min_hits, max_cosine_distance, embedder, embedder_gpu):
    ds = DeepSort(
        max_age=max_age, n_init=min_hits, nms_max_overlap=1.0,
        max_cosine_distance=max_cosine_distance, nn_budget=None,
        embedder=embedder, half=True, bgr=True, embedder_gpu=embedder_gpu,
    )
    return DeepSortWrapper(ds)


def update_tracker(tracker, detections, frame=None):
    if detections is None or len(detections) == 0:
        detections = np.empty((0, 6))
    return tracker.update(detections, frame)


def draw_tracks(frame, tracker, tracked_dets, class_names, line_thickness=2, id_map=None):
    if len(tracked_dets) > 0:
        for i, box in enumerate(tracked_dets[:, :4]):
            x1, y1, x2, y2 = [int(v) for v in box]
            tid        = int(tracked_dets[i, 8])
            cid        = int(tracked_dets[i, 4])
            name       = class_names.get(cid + 1, str(cid))
            display_id = id_map.get(tid, tid) if id_map else tid
            label      = f"{display_id} {name}"
            (tw, _), _ = cv2.getTextSize(label, cv2.FONT_HERSHEY_SIMPLEX, 0.6, 1)
            cv2.rectangle(frame, (x1, y1), (x2, y2), (0, 255, 253), line_thickness)
            cv2.rectangle(frame, (x1, y1 - 20), (x1 + tw, y1), (255, 144, 30), -1)
            cv2.putText(frame, label, (x1, y1 - 5),
                        cv2.FONT_HERSHEY_SIMPLEX, 0.6, (255, 255, 255), 1)
    return frame


print("DeepSORT tracker ready. It will follow clothing items across frames using motion and appearance.")
print("Step 8 complete. Proceed to Step 9.")

## Step 9 — Define the Main Pipeline Function

This is the heart of the system, defined here but not yet executed. When called in Step 10, it runs the full pipeline sequence from the diagram: it opens the video (Video Ingestion), runs the YOLO model on each frame to detect clothing items (Object Detection), feeds those detections into DeepSORT (Tracking), applies global re-identification to maintain stable IDs across the whole video (Global Re-identification), selects the largest and sharpest crop seen for each clothing item (Cropping and Best Frame Selection), filters out blurry and near-duplicate crops (Crop Quality Enhancement), and writes the final JSON output with timestamps and metadata for each clothing item (Output Metadata Generation). All the components from Steps 6 through 8 come together here.

In [ ]:
from ultralytics import YOLO
import matplotlib
matplotlib.use("Agg")   # headless backend — prevents display errors on HPC
import matplotlib.pyplot as plt

_MLFLOW_STEP_INTERVAL = 30   # log per-frame metrics every N frames


def run_pipeline(
    video_path, model_path, device, save_video,
    conf, iou,
    deepsort_max_age, deepsort_n_init, deepsort_max_cosine, deepsort_embedder, embedder_gpu,
    use_global_ids, gid_max_gap_s, gid_cosine_threshold, gid_spatial_gate_pps,
    blur_threshold, phash_threshold, min_visible_s, line_thickness,
    log_dir, detections_dir, output_dir,
    mlflow_tracking_uri=None, mlflow_experiment="ClickTheLook",
):
    print(f"Loading model : {model_path}")
    model = YOLO(model_path)

    tracker_config = {
        "max_age": deepsort_max_age, "min_hits": deepsort_n_init,
        "max_cosine_distance": deepsort_max_cosine,
        "embedder": deepsort_embedder, "embedder_gpu": embedder_gpu,
    }
    tracker = init_tracker(**tracker_config)
    print("Tracker       : DeepSORT")

    cap = cv2.VideoCapture(video_path)
    if not cap.isOpened():
        raise RuntimeError(f"Cannot open: {video_path}")
    src_fps      = cap.get(cv2.CAP_PROP_FPS) or 30
    w            = int(cap.get(cv2.CAP_PROP_FRAME_WIDTH))
    h            = int(cap.get(cv2.CAP_PROP_FRAME_HEIGHT))
    total_frames = int(cap.get(cv2.CAP_PROP_FRAME_COUNT))
    print(f"Source        : {video_path}  ({w}x{h} @ {src_fps:.1f} FPS, {total_frames} frames)")

    writer = None
    if save_video:
        fourcc = cv2.VideoWriter_fourcc(*"mp4v")
        writer = cv2.VideoWriter(save_video, fourcc, src_fps, (w, h))
        print(f"Saving video  : {save_video}")

    logger = RunLogger(
        model_path=model_path, model_info=extract_model_info(model),
        tracker_backend="deepsort", tracker_config=tracker_config,
        source=video_path,
        source_meta={"type": "video_file", "width": w, "height": h,
                     "source_fps": src_fps, "total_frames_src": total_frames,
                     "duration_s": round(total_frames / src_fps, 2) if src_fps > 0 else None},
        conf=conf, iou=iou, device=device, save_video=save_video, log_dir=log_dir,
    )
    print(f"Run ID        : {logger.run_id}")

    active_run = None
    if mlflow_tracking_uri:
        mlflow.set_tracking_uri(mlflow_tracking_uri)
        mlflow.set_experiment(mlflow_experiment)
        active_run = mlflow.start_run(run_name=f"ClickTheLook_{logger.run_id}")
        mlflow.log_params({
            "model_path":           str(model_path),
            "device":               str(device),
            "conf":                 conf,
            "iou":                  iou,
            "deepsort_max_age":     deepsort_max_age,
            "deepsort_n_init":      deepsort_n_init,
            "deepsort_max_cosine":  deepsort_max_cosine,
            "deepsort_embedder":    deepsort_embedder,
            "use_global_ids":       use_global_ids,
            "gid_max_gap_s":        gid_max_gap_s,
            "gid_cosine_threshold": gid_cosine_threshold,
            "gid_spatial_gate_pps": gid_spatial_gate_pps,
            "blur_threshold":       blur_threshold,
            "phash_threshold":      phash_threshold,
            "min_visible_s":        min_visible_s,
            "video_fps":            src_fps,
            "video_resolution":     f"{w}x{h}",
            "total_frames":         total_frames,
        })
        print(f"MLflow run    : {active_run.info.run_id}")

    gid_manager = None
    if use_global_ids:
        gid_manager = GlobalIdentityManager(
            max_gap_s=gid_max_gap_s,
            cosine_threshold=gid_cosine_threshold,
            spatial_gate_pps=gid_spatial_gate_pps,
        )
        print("Global re-ID  : enabled")

    frame_idx    = 0
    t_start      = time.time()
    best_crops   = {}
    id_map       = {}
    blurry_count = 0
    near_dupes   = 0

    try:
        while True:
            ret, frame = cap.read()
            if not ret:
                break
            frame_idx += 1

            t0      = time.perf_counter()
            results = model.predict(frame, conf=conf, iou=iou, device=device, verbose=False)
            t_infer = (time.perf_counter() - t0) * 1000.0
            boxes   = results[0].boxes
            if boxes is not None and len(boxes) > 0:
                dets = np.hstack([boxes.xyxy.cpu().numpy(),
                                  boxes.conf.cpu().numpy().reshape(-1, 1),
                                  boxes.cls.cpu().numpy().reshape(-1, 1)])
            else:
                dets = np.empty((0, 6))

            t1      = time.perf_counter()
            tracked = update_tracker(tracker, dets, frame)
            t_track = (time.perf_counter() - t1) * 1000.0

            if gid_manager is not None:
                embeddings = tracker.get_embeddings() if hasattr(tracker, "get_embeddings") else {}
                id_map     = gid_manager.update(tracked, embeddings, CATEGORIES, time.time())
            else:
                id_map = {}

            if len(tracked) > 0:
                fh, fw = frame.shape[:2]
                for row in tracked:
                    x1c  = max(0, int(row[0]));  y1c = max(0, int(row[1]))
                    x2c  = min(fw, int(row[2])); y2c = min(fh, int(row[3]))
                    tid  = int(row[8])
                    gid  = id_map.get(tid, tid)
                    area = (x2c - x1c) * (y2c - y1c)
                    if area > best_crops.get(gid, {}).get("area", -1) and area > 0:
                        best_crops[gid] = {
                            "area":       area,
                            "crop":       frame[y1c:y2c, x1c:x2c].copy(),
                            "class_name": CATEGORIES.get(int(row[4]) + 1, str(int(row[4]))),
                        }

            t2        = time.perf_counter()
            annotated = draw_tracks(frame, tracker, tracked, CATEGORIES,
                                    line_thickness=line_thickness, id_map=id_map or None)
            t_draw    = (time.perf_counter() - t2) * 1000.0

            logger.log_frame(frame_idx=frame_idx, inference_ms=t_infer,
                             tracker_ms=t_track, draw_ms=t_draw,
                             detections=dets, tracked=tracked,
                             class_names=CATEGORIES, id_map=id_map or None)

            if active_run and frame_idx % _MLFLOW_STEP_INTERVAL == 0:
                elapsed     = time.time() - t_start
                live_fps    = frame_idx / elapsed if elapsed > 0 else 0
                total_ms    = t_infer + t_track + t_draw
                active_gids = len(set(id_map.values())) if id_map else len(tracked)
                avg_conf    = float(np.mean(dets[:, 4])) if len(dets) > 0 else 0.0
                mlflow.log_metric("detections_per_frame",   len(dets),          step=frame_idx)
                mlflow.log_metric("tracks_per_frame",       len(tracked),       step=frame_idx)
                mlflow.log_metric("active_identities_live", active_gids,        step=frame_idx)
                mlflow.log_metric("avg_confidence_live",    round(avg_conf, 3), step=frame_idx)
                mlflow.log_metric("inference_ms_live",      round(t_infer,  1), step=frame_idx)
                mlflow.log_metric("tracker_ms_live",        round(t_track,  1), step=frame_idx)
                mlflow.log_metric("draw_ms_live",           round(t_draw,   1), step=frame_idx)
                mlflow.log_metric("pipeline_ms_live",       round(total_ms, 1), step=frame_idx)
                mlflow.log_metric("live_fps",               round(live_fps, 1), step=frame_idx)

            elapsed  = time.time() - t_start
            live_fps = frame_idx / elapsed if elapsed > 0 else 0
            cv2.putText(annotated, f"FPS: {live_fps:.1f}  Frame: {frame_idx}",
                        (10, 30), cv2.FONT_HERSHEY_SIMPLEX, 0.8, (0, 255, 0), 2)

            if writer is not None:
                writer.write(annotated)

            if total_frames > 0 and frame_idx % 100 == 0:
                pct = frame_idx / total_frames * 100
                print(f"  {frame_idx}/{total_frames} ({pct:.0f}%)  FPS: {live_fps:.1f}")

    except KeyboardInterrupt:
        print("Interrupted.")
    finally:
        cap.release()
        if writer is not None:
            writer.release()
        elapsed = time.time() - t_start
        avg_fps = frame_idx / elapsed if elapsed > 0 else 0
        print(f"\nDone. {frame_idx} frames in {elapsed:.1f}s ({avg_fps:.1f} FPS avg)")

        final_crops              = {}
        total_identities_tracked = len(best_crops)

        if best_crops:
            valid = {g: i for g, i in best_crops.items()
                     if i["crop"] is not None and i["crop"].size > 0}
            blurry_count = sum(1 for i in valid.values() if _blur_score(i["crop"]) < blur_threshold)
            valid        = {g: i for g, i in valid.items() if _blur_score(i["crop"]) >= blur_threshold}
            candidates   = sorted(valid.items(), key=lambda x: x[1]["area"], reverse=True)
            kept_hashes  = []
            for gid, info in candidates:
                h = _dhash(info["crop"])
                if any(bin(h ^ kh).count("1") < phash_threshold for kh in kept_hashes):
                    near_dupes += 1
                else:
                    kept_hashes.append(h)
                    final_crops[gid] = info

            det_dir = Path(detections_dir) / logger.run_id
            det_dir.mkdir(parents=True, exist_ok=True)
            for gid, info in final_crops.items():
                cv2.imwrite(str(det_dir / f"{gid}_{info['class_name']}.jpg"), info["crop"])

            parts = [f"{len(final_crops)} saved"]
            if blurry_count: parts.append(f"{blurry_count} blurry removed")
            if near_dupes:   parts.append(f"{near_dupes} near-duplicate(s) removed")
            print(f"Crops    : {', '.join(parts)} -> {det_dir}")

        class_tracked_counts = {}
        for info in best_crops.values():
            c = info["class_name"]
            class_tracked_counts[c] = class_tracked_counts.get(c, 0) + 1

        class_quality_counts = {}
        for info in final_crops.values():
            c = info["class_name"]
            class_quality_counts[c] = class_quality_counts.get(c, 0) + 1

        log_path = logger.finalize(frame_idx, elapsed)
        print(f"Run log  : {log_path}")

        det_dir  = Path(detections_dir) / logger.run_id
        out_data = []
        for entry in logger.identity_summary:
            gid       = entry["id"]
            cls       = entry["class"]
            crop_file = det_dir / f"{gid}_{cls}.jpg"
            if not crop_file.exists():
                continue
            if entry["total_visible_seconds"] < min_visible_s:
                continue
            out_data.append({
                "id":                    gid,
                "class":                 cls,
                "crop_path":             str(crop_file),
                "num_intervals":         len(entry["intervals"]),
                "intervals":             entry["intervals"],
                "total_visible_seconds": entry["total_visible_seconds"],
                "total_visible_text":    entry["total_visible_text"],
            })
        out_path = Path(output_dir) / f"{logger.run_id}.json"
        out_path.parent.mkdir(parents=True, exist_ok=True)
        with open(out_path, "w") as fh:
            json.dump(out_data, fh, indent=2)
        print(f"Output   : {out_path}")

        class_out_counts = {}
        class_vis_data   = {}
        for entry in out_data:
            c = entry["class"]
            class_out_counts[c] = class_out_counts.get(c, 0) + 1
            class_vis_data.setdefault(c, []).append(entry["total_visible_seconds"])
        avg_vis_per_class = {
            c: round(float(np.mean(vals)), 2) for c, vals in class_vis_data.items()
        }

        logger.class_tracked_counts = class_tracked_counts
        logger.class_quality_counts = class_quality_counts
        logger.class_out_counts     = class_out_counts
        logger.avg_vis_per_class    = avg_vis_per_class

        if active_run:
            perf = logger._meta.get("performance", {})
            det  = logger._meta.get("detections",  {})
            trk  = logger._meta.get("tracking",    {})

            identities_after_quality    = len(final_crops)
            identities_excluded_quality = total_identities_tracked - identities_after_quality
            identities_excluded_vis     = identities_after_quality - len(out_data)

            total_tracker_ids   = trk.get("total_unique_ids", 0)
            trk_lifespan        = trk.get("track_lifespan_frames", {})
            track_summary_list  = trk.get("track_summary", [])
            fragmentation_ratio = round(total_tracker_ids / max(total_identities_tracked, 1), 3)
            reid_events         = max(0, total_tracker_ids - total_identities_tracked)
            avg_kalman_gap      = round(float(np.mean([t["kalman_gap_frames"] for t in track_summary_list])), 2) \
                                  if track_summary_list else 0.0
            track_stability_rate = round(1.0 / max(fragmentation_ratio, 1e-6), 3)

            if final_crops:
                blur_scores_saved = [_blur_score(i["crop"]) for i in final_crops.values()]
                avg_blur_saved    = round(float(np.mean(blur_scores_saved)), 2)
                min_blur_saved    = round(float(np.min(blur_scores_saved)),  2)
                avg_crop_area_px  = round(float(np.mean([i["area"] for i in final_crops.values()])), 0)
            else:
                avg_blur_saved = min_blur_saved = avg_crop_area_px = 0.0
            blur_removal_rate = round(blurry_count / max(total_identities_tracked, 1), 3)
            dup_removal_rate  = round(near_dupes   / max(total_identities_tracked, 1), 3)

            if out_data:
                vis_secs      = [e["total_visible_seconds"] for e in out_data]
                intervals_cnt = [e["num_intervals"] for e in out_data]
                avg_vis  = round(float(np.mean(vis_secs)),           2)
                max_vis  = round(float(np.max(vis_secs)),            2)
                min_vis  = round(float(np.min(vis_secs)),            2)
                p50_vis  = round(float(np.percentile(vis_secs, 50)), 2)
                p90_vis  = round(float(np.percentile(vis_secs, 90)), 2)
                avg_ivls = round(float(np.mean(intervals_cnt)),      2)
                multi_iv = sum(1 for x in intervals_cnt if x > 1)
                single_iv= sum(1 for x in intervals_cnt if x == 1)
            else:
                avg_vis=max_vis=min_vis=p50_vis=p90_vis=avg_ivls=multi_iv=single_iv=0

            per_class_dets   = det.get("per_class_detections", {})
            per_class_tracks = det.get("per_class_tracked_frames", {})

            mlflow.log_metrics({
                "avg_fps":                        perf.get("realtime_avg_fps", 0),
                "frames_processed":               perf.get("frames_processed", 0),
                "avg_pipeline_ms":                perf.get("total_pipeline_ms", {}).get("avg", 0),
                "p95_pipeline_ms":                perf.get("total_pipeline_ms", {}).get("p95", 0),
                "avg_inference_ms":               perf.get("inference_ms", {}).get("avg", 0),
                "p95_inference_ms":               perf.get("inference_ms", {}).get("p95", 0),
                "avg_tracker_ms":                 perf.get("tracker_ms", {}).get("avg", 0),
                "avg_draw_ms":                    perf.get("draw_ms", {}).get("avg", 0),
                "zero_detection_rate":            perf.get("zero_detection_rate", 0),
                "frames_with_zero_detections":    perf.get("frames_with_zero_dets", 0),
                "detection_coverage_pct":         round((1 - perf.get("zero_detection_rate", 0)) * 100, 2),
                "total_detections":               det.get("total", 0),
                "avg_detections_per_frame":       det.get("per_frame", {}).get("avg", 0),
                "max_detections_in_frame":        det.get("per_frame", {}).get("max", 0),
                "p95_detections_per_frame":       det.get("per_frame", {}).get("p95", 0),
                "avg_confidence":                 det.get("confidence", {}).get("avg", 0),
                "min_confidence":                 det.get("confidence", {}).get("min", 0),
                "p50_confidence":                 det.get("confidence", {}).get("p50", 0),
                "total_unique_tracker_ids":       total_tracker_ids,
                "avg_tracks_per_frame":           trk.get("tracks_per_frame", {}).get("avg", 0),
                "max_tracks_in_frame":            trk.get("tracks_per_frame", {}).get("max", 0),
                "avg_track_lifespan_frames":      trk_lifespan.get("avg", 0),
                "p50_track_lifespan_frames":      trk_lifespan.get("p50", 0),
                "max_track_lifespan_frames":      trk_lifespan.get("max", 0),
                "track_fragmentation_ratio":      fragmentation_ratio,
                "track_stability_rate":           track_stability_rate,
                "reid_events":                    reid_events,
                "avg_kalman_gap_frames":          avg_kalman_gap,
                "identities_total_tracked":       total_identities_tracked,
                "identities_removed_blurry":      blurry_count,
                "identities_removed_duplicate":   near_dupes,
                "identities_after_quality":       identities_after_quality,
                "identities_excluded_quality":    identities_excluded_quality,
                "identities_excluded_visibility": identities_excluded_vis,
                "identities_in_output":           len(out_data),
                "avg_blur_score_saved":           avg_blur_saved,
                "min_blur_score_saved":           min_blur_saved,
                "avg_crop_area_px":               avg_crop_area_px,
                "blur_removal_rate":              blur_removal_rate,
                "duplicate_removal_rate":         dup_removal_rate,
                "crops_saved":                    identities_after_quality,
                "avg_visibility_seconds":         avg_vis,
                "max_visibility_seconds":         max_vis,
                "min_visibility_seconds":         min_vis,
                "p50_visibility_seconds":         p50_vis,
                "p90_visibility_seconds":         p90_vis,
                "avg_intervals_per_identity":     avg_ivls,
                "multi_interval_identities":      multi_iv,
                "single_interval_identities":     single_iv,
                "reid_effectiveness_pct":         round(multi_iv / max(len(out_data), 1) * 100, 2),
            })

            for c, v in per_class_dets.items():
                mlflow.log_metric(f"cls_{c}_detections",          v)
            for c, v in per_class_tracks.items():
                mlflow.log_metric(f"cls_{c}_track_frames",        v)
            for c, v in class_tracked_counts.items():
                mlflow.log_metric(f"cls_{c}_identities_tracked",  v)
            for c, v in class_quality_counts.items():
                mlflow.log_metric(f"cls_{c}_identities_quality",  v)
            for c, v in class_out_counts.items():
                mlflow.log_metric(f"cls_{c}_in_output",           v)
            for c, v in avg_vis_per_class.items():
                mlflow.log_metric(f"cls_{c}_avg_visibility_s",    v)

            mlflow.log_artifact(log_path)
            mlflow.log_artifact(str(out_path))
            logger.mlflow_run_id = active_run.info.run_id
            mlflow.end_run()
            print(f"MLflow   : run {active_run.info.run_id}  (experiment: {mlflow_experiment})")

    return logger


print("Pipeline function defined and ready.")
print("Step 9 complete. Proceed to Step 10 to start processing.")

## Step 10 — Run the Pipeline

This cell starts the full pipeline using all the settings from Step 2. The model and video are loaded, and each frame is processed one by one. A progress update is printed every 100 frames showing how far along the video is and the current processing speed. When the video is finished, all output files are saved automatically.

In [ ]:
logger = run_pipeline(
    video_path           = VIDEO_PATH,
    model_path           = MODEL_PATH,
    device               = DEVICE,
    save_video           = SAVE_VIDEO,
    conf                 = CONF,
    iou                  = IOU,
    deepsort_max_age     = DEEPSORT_MAX_AGE,
    deepsort_n_init      = DEEPSORT_N_INIT,
    deepsort_max_cosine  = DEEPSORT_MAX_COSINE,
    deepsort_embedder    = DEEPSORT_EMBEDDER,
    embedder_gpu         = EMBEDDER_GPU,
    use_global_ids       = USE_GLOBAL_IDS,
    gid_max_gap_s        = GID_MAX_GAP_S,
    gid_cosine_threshold = GID_COSINE_THRESHOLD,
    gid_spatial_gate_pps = GID_SPATIAL_GATE_PPS,
    blur_threshold       = BLUR_THRESHOLD,
    phash_threshold      = PHASH_THRESHOLD,
    min_visible_s        = MIN_VISIBLE_S,
    line_thickness       = LINE_THICKNESS,
    log_dir              = LOG_DIR,
    detections_dir       = DETECTIONS_DIR,
    output_dir           = OUTPUT_DIR,
    mlflow_tracking_uri  = MLFLOW_TRACKING_URI,
    mlflow_experiment    = MLFLOW_EXPERIMENT,
)

## Step 11 — View the Results Summary

After the pipeline finishes, this cell reads the output JSON file and prints a clean summary of the run: total frames processed, average processing speed in frames per second, which stage was the bottleneck, and a table listing every clothing item that passed all quality filters along with how many times it appeared and for how long.

In [ ]:
out_path = sorted(OUTPUT_DIR.glob("*.json"))[-1]
data     = json.load(open(out_path))
perf     = logger._meta.get("performance", {})

print(f"Output file     : {out_path.name}")
print(f"Frames processed: {perf.get('frames_processed')}")
print(f"Avg FPS         : {perf.get('realtime_avg_fps')}")
print(f"Bottleneck      : {perf.get('bottleneck')}")
print(f"Inference avg   : {perf.get('inference_ms', {}).get('avg')} ms")
print(f"Identities out  : {len(data)}")
if hasattr(logger, "mlflow_run_id"):
    print(f"\nMLflow run ID   : {logger.mlflow_run_id}")
    print(f"View UI with    : mlflow ui --backend-store-uri {MLFLOW_TRACKING_URI}")
print()
print(f"{'ID':>5}  {'Class':<24} {'Intervals':>9}  {'Visible':>8}")
print("-" * 55)
for entry in data:
    n = entry['num_intervals']
    print(f"{entry['id']:>5}  {entry['class']:<24} {n:>8}x  {entry['total_visible_text']:>8}")
print()
print(f"Step 11 complete. {len(data)} clothing item(s) in the output. Proceed to Step 12 to view the crops.")

## Step 12 — View the Detected Clothing Crops

This cell displays all the crop images the pipeline saved, laid out in a grid. Each image shows the item's assigned ID, its clothing category, how many separate intervals it was visible in the video, and the total time it appeared. The grid is also saved as an image file to your output folder.

In [ ]:
import matplotlib.pyplot as plt
import matplotlib.image as mpimg

entries = [e for e in data if e.get("crop_path") and Path(e["crop_path"]).exists()]

if not entries:
    print("No crops found to save.")
else:
    cols = min(5, len(entries))
    rows = (len(entries) + cols - 1) // cols
    fig, axes = plt.subplots(rows, cols, figsize=(cols * 3, rows * 4))
    axes = axes.flatten() if len(entries) > 1 else [axes]

    for i, entry in enumerate(entries):
        axes[i].imshow(mpimg.imread(entry["crop_path"]))
        axes[i].set_title(
            f"ID {entry['id']}  {entry['class']}\n"
            f"{entry['num_intervals']}x interval  {entry['total_visible_text']}",
            fontsize=8)
        axes[i].axis("off")
    for j in range(len(entries), len(axes)):
        axes[j].axis("off")

    plt.suptitle(f"Detected Clothing — {len(entries)} items", fontsize=12, y=1.01)
    plt.tight_layout()
    grid_path = OUTPUT_DIR / f"{out_path.stem}_crops_grid.png"
    plt.savefig(grid_path, bbox_inches="tight", dpi=150)
    plt.close()
    print(f"Crops grid saved : {grid_path}")

    if hasattr(logger, "mlflow_run_id"):
        mlflow.set_tracking_uri(MLFLOW_TRACKING_URI)
        with mlflow.start_run(run_id=logger.mlflow_run_id):
            mlflow.log_artifact(str(grid_path))
        print(f"Crops grid logged to MLflow run {logger.mlflow_run_id}")

print(f"Output JSON saved : {out_path}")
print(f"\nStep 12 complete. {len(entries) if entries else 0} crop image(s) saved.")